In [ ]:
# Machine-specific paths live in config.yaml next to this notebook (gitignored).
# Copy config.example.yaml -> config.yaml and fill in your paths.
from phoreal.config import load_config
cfg = load_config('config.yaml')

### Import packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters ---
sigma = 2.79 # From your empirical fit
spacing = 0.7 # ICESat-2 along-track firing rate (m)
track_length = 30 # Simulate a 30m stretch of flight
fwhm_radius = (sigma * 2.355) / 2
beam_radius = (sigma * 4) / 2

# Create a high-res spatial grid
x = np.linspace(-15, 15, 500) # Along-track
y = np.linspace(-15, 15, 500) # Cross-track
X, Y = np.meshgrid(x, y)

# --- Simulate the Overlapping Pulses ---
accumulated_energy = np.zeros_like(X)
pulse_centers = np.arange(-track_length/2, track_length/2, spacing)

# Sum the 2D Gaussian pulses
for center in pulse_centers:
    # 2D Gaussian equation
    pulse = np.exp(-((X - center)**2 + Y**2) / (2 * sigma**2))
    accumulated_energy += pulse

# Normalize so the dead-center gets exactly 100% relative power
max_energy = np.max(accumulated_energy)
accumulated_energy_pct = (accumulated_energy / max_energy) * 100

# Extract a 1D cross-track profile (take a slice at X=0)
cross_track_profile = accumulated_energy_pct[:, 250] 

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [1.5, 1]})

# Top Panel: 2D Heatmap
c1 = ax1.contourf(X, Y, accumulated_energy_pct, levels=100, cmap='Greens', alpha=0.9)

# ---> NEW: Add Colorbar <---
cbar = fig.colorbar(c1, ax=ax1)
cbar.set_label('Accumulated Power (%)', fontsize=12)

# Boundary Lines
ax1.axhline(0, color='black', linestyle='-', alpha=0.5, label='Flight Path Center')
ax1.axhline(fwhm_radius, color='red', linestyle='--', alpha=0.5, label='FWHM Edge')
ax1.axhline(-fwhm_radius, color='red', linestyle='--', alpha=0.5)
ax1.axhline(beam_radius, color='black', linestyle='--', alpha=0.5, label='Beam Edge')
ax1.axhline(-beam_radius, color='black', linestyle='--', alpha=0.5)

# Highlight our two hypothetical trees
tree_center_y, tree_flank_y = 0.0, 5.0
ax1.scatter([0], [tree_center_y], color='red', s=100, edgecolors='white', zorder=5, label='Center Tree (0m)')
ax1.scatter([0], [tree_flank_y], color='blue', s=100, edgecolors='white', zorder=5, label='Flank Tree (5m)')

ax1.set_title(f'ICESat-2 Overlapping Beam Track ($\sigma$ = {sigma}m, spacing = {spacing}m)', fontsize=14)
ax1.set_ylabel('Crosstrack Distance (m)', fontsize=12)
ax1.set_xlabel('Alongtrack Distance (m)', fontsize=12)
ax1.set_xlim(-10, 10)
ax1.set_ylim(-12, 12)

# Moved to lower right with an opaque frame to prevent line overlap
ax1.legend(loc='lower right', framealpha=0.9)

# Bottom Panel: 1D Relative Power Profile
ax2.plot(y, cross_track_profile, color='forestgreen', linewidth=3)
ax2.fill_between(y, cross_track_profile, color='forestgreen', alpha=0.2)

# Mark the energy received by the two trees
energy_center = cross_track_profile[np.abs(y - tree_center_y).argmin()]
energy_flank = cross_track_profile[np.abs(y - tree_flank_y).argmin()]

ax2.axvline(fwhm_radius, color='red', linestyle='--', alpha=0.5, label='FWHM Edge')
ax2.axvline(-fwhm_radius, color='red', linestyle='--', alpha=0.5)
ax2.axvline(beam_radius, color='black', linestyle='--', alpha=0.5, label='Beam Edge')
ax2.axvline(-beam_radius, color='black', linestyle='--', alpha=0.5)

ax2.scatter([tree_center_y], [energy_center], color='red', s=100, zorder=5)
ax2.scatter([tree_flank_y], [energy_flank], color='blue', s=100, zorder=5)

# Annotate the drop-off
ax2.annotate(f"{energy_center:.1f}% Energy", xy=(tree_center_y, energy_center), 
             xytext=(tree_center_y + 0.5, energy_center - 10), fontsize=12, color='red')
ax2.annotate(f"{energy_flank:.1f}% Energy", xy=(tree_flank_y, energy_flank), 
             xytext=(tree_flank_y + 0.5, energy_flank + 10), fontsize=12, color='blue')

ax2.legend(loc='upper right', framealpha=0.9)
ax2.set_title('Relative Accumulated Energy Profile', fontsize=14)
ax2.set_xlabel('Crosstrack Distance (m)', fontsize=12)
ax2.set_ylabel('Accumulated Power (%)', fontsize=12)
ax2.set_xlim(-12, 12)
ax2.set_ylim(0, 110)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# --- Data & Math ---
# Radii
w_fwhm = 6.56
w_total = 11.18
r_core = w_fwhm/2 # 7.5m diameter FWHM
r_total = w_total/2 # 11.0m diameter boundary

# Calculate Physical Areas (m^2)
area_core = np.pi * r_core**2
area_total = np.pi * r_total**2
area_wings = area_total - area_core # The 'donut' shape

# The calculated accumulated energy percentages
energy_core = 50
energy_wings = 50

# Calculate Energy Density (% of total energy per square meter)
# This is our proxy for "Photon Probability"
density_core = energy_core / area_core
density_wings = energy_wings / area_wings

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1, 1]})

# --- Left Plot: Footprint Geometry ---
ax1.set_aspect('equal')
ax1.set_xlim(-6.5, 6.5)
ax1.set_ylim(-6.5, 6.5)
ax1.axis('off')

# Outer Ring (Wings)
wing_circle = patches.Circle((0, 0), r_total, facecolor='#a1d99b', edgecolor='#31a354', alpha=0.5, linestyle='--', linewidth=2)
ax1.add_patch(wing_circle)

# Inner Core (FWHM)
core_circle = patches.Circle((0, 0), r_core, facecolor='#31a354', edgecolor='#006d2c', alpha=0.9, linewidth=2)
ax1.add_patch(core_circle)

# Annotations for Left Plot
ax1.annotate(f'FWHM Core\n{energy_core}% Energy\n{area_core:.1f} m$^2$ Area', 
             xy=(0, 0), ha='center', va='center', color='white', fontsize=14, fontweight='bold')

ax1.annotate(f'Outer Wings\n{energy_wings}% Energy\n{area_wings:.1f} m$^2$ Area', 
             xy=(0, 4.5), ha='center', va='center', color='#006d2c', fontsize=14, fontweight='bold')

ax1.set_title(f"Spatial Distribution: Energy vs. Area\nFor a {r_total*2} m footprint", fontsize=16, fontweight='bold', pad=15)

# --- Right Plot: Energy Density Bar Chart ---
labels = [f'FWHM Core\n({w_fwhm}m Dia.)', f'Outer Wings\n({w_total}m Dia. Boundary)']
densities = [density_core, density_wings]
colors = ['#31a354', '#a1d99b']

bars = ax2.bar(labels, densities, color=colors, edgecolor='#006d2c', width=0.6)
ax2.set_ylabel('Energy Density (% of Total Energy per m$^2$)', fontsize=14)
ax2.set_title('Photon Probability (Energy Density)', fontsize=16, fontweight='bold', pad=15)
ax2.set_ylim(0, 2.5)
ax2.grid(axis='y', linestyle='--', alpha=0.7)
ax2.tick_params(axis='x', labelsize=13)

# Add text on top of bars
for bar, density in zip(bars, densities):
    height = bar.get_height()
    ax2.annotate(f'{density:.2f}% / m$^2$',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 8),  
                 textcoords="offset points",
                 ha='center', va='bottom', fontsize=14, fontweight='bold', color='#006d2c')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# --- The "Ribbon" Math ---
width_core = 6.56 # FWHM width
width_total = 11.18 # Standard footprint boundary
width_wings = width_total - width_core # Combined width of left and right flanks

# Accumulated energy percentages based on the 1D Gaussian integration
energy_core = 76.1
energy_wings = 19.3 # Energy between FWHM and 11m boundary

# Density (% of total energy per cross-track meter)
density_core = energy_core / width_core
density_wings = energy_wings / width_wings

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1, 1]})

# --- Left Plot: Swath Geometry (The Ribbon) ---
ax1.set_xlim(-6.5, 6.5)
ax1.set_ylim(-10, 10)
ax1.axis('off')

# Left Wing
left_wing = patches.Rectangle((-5.5, -10), 2.25, 20, facecolor='#a1d99b', edgecolor='#31a354', alpha=0.5, linestyle='--')
ax1.add_patch(left_wing)

# Right Wing
right_wing = patches.Rectangle((3.25, -10), 2.25, 20, facecolor='#a1d99b', edgecolor='#31a354', alpha=0.5, linestyle='--')
ax1.add_patch(right_wing)

# Center Core
core_swath = patches.Rectangle((-3.25, -10), 6.5, 20, facecolor='#31a354', edgecolor='#006d2c', alpha=0.9)
ax1.add_patch(core_swath)

# Annotations for Left Plot
ax1.annotate(f'FWHM Core Track\n{energy_core}% Energy\n({width_core}m wide)', 
             xy=(0, 0), ha='center', va='center', color='white', fontsize=13, fontweight='bold')

ax1.annotate(f'Flank', xy=(-4.375, 4), ha='center', va='center', color='#006d2c', fontsize=12, fontweight='bold', rotation=90)
ax1.annotate(f'Flank', xy=(4.375, 4), ha='center', va='center', color='#006d2c', fontsize=12, fontweight='bold', rotation=270)

ax1.set_title("Accumulated Flight Path Swath", fontsize=16, fontweight='bold', pad=15)

# --- Right Plot: Energy Density Bar Chart ---
labels = ['FWHM Core Track\n(6.5m Swath)', 'Outer Flanks\n(Remaining 11m Swath)']
densities = [density_core, density_wings]
colors = ['#31a354', '#a1d99b']

bars = ax2.bar(labels, densities, color=colors, edgecolor='#006d2c', width=0.6)
ax2.set_ylabel('Avg. Energy Density (% of Total Energy / cross-track meter)', fontsize=12)
ax2.set_title('Accumulated Photon Probability', fontsize=16, fontweight='bold', pad=15)
ax2.set_ylim(0, 14)
ax2.grid(axis='y', linestyle='--', alpha=0.7)
ax2.tick_params(axis='x', labelsize=13)

# Add text on top of bars
for bar, density in zip(bars, densities):
    height = bar.get_height()
    ax2.annotate(f'{density:.1f}% / m',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 8),  
                 textcoords="offset points",
                 ha='center', va='bottom', fontsize=14, fontweight='bold', color='#006d2c')

# Add the ratio text highlighting the 2.7x density difference
ratio = density_core / density_wings
# ax2.annotate(f"Core is ~{ratio:.1f}x more dense", xy=(0.5, 12), ha='center', fontsize=14, color='red', fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="red", lw=2))

plt.tight_layout()
# Optional: save the figure to your directory
# plt.savefig('ribbon_energy_density.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
%load_ext autoreload
%autoreload 2

#Import general libs you'll need
import os
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import numpy as np

# Import functions from main
from phoreal.agg_configs import ATL_AGG_CONFIG, ALS_AGG_CONFIG, ATL08_AGG_CONFIG

# Import functions from the utils
from phoreal.io import readers, processing
from phoreal import analysis
from phoreal import plotting as plotter
from phoreal.als.swath import create_als_swath, transform_als_swath
from phoreal.geo.datum_transforms import convert_3d_nad83_to_wgs84
from phoreal.geo.datum_transforms import get_geoid_height



### Identify files



In [ ]:
atl03_dir = cfg['austin_atl_dir']
atl03_name = 'ATL03_20250616013703_00052802_007_01.h5'
atl03_name = 'ATL03_20181215070448_11860106_007_01.h5'
atl03_file = os.path.join(atl03_dir, atl03_name)

atl08_dir = cfg['austin_atl_dir']
atl08_name = 'ATL08_20250616013703_00052802_007_01.h5'
atl08_name = 'ATL08_20181215070448_11860106_007_01.h5'
atl08_file = os.path.join(atl08_dir, atl08_name)

# Define the ground track of interest
gt = 'gt2r'

# Identify ATX AOI
extent_file = cfg['austin_extent_file']

# ALS Geoid File
als_geoid_file = cfg['als_geoid_file']

### Get photon rate dataframe


In [ ]:
# Get photon rate DF
df_ph = readers.read_photon_dataframe(atl03_file, gt, atl08_file)


In [ ]:
df_ph['crosstrack'] = 0 

In [ ]:
# Filter df_ph
extent_gdf = gpd.read_file(extent_file)
df_ph = processing.filter_df_by_extent(df_ph, extent_gdf.total_bounds)


### Plot output 

Check to make sure it plots correctly

In [ ]:
import matplotlib.pyplot as plt
plotter.plot_alongtrack(df_ph.alongtrack, df_ph.h_ph, df_ph.atl08_class,'Example Plot')

### Process ALS Swath

Load the ALS Swath and preprocess it 

In [ ]:
atl03_basename = os.path.basename(atl03_name).split('.h5')[0]
print(atl03_basename)

In [ ]:
# Load swath if available
als_outdir = cfg['als_swath_dir']
file_out_name = f"{os.path.basename(atl03_name).split('.h5')[0]}_{gt}"
als_outfile = os.path.join(als_outdir, f'als_{file_out_name}.pqt')                
if os.path.exists(als_outfile):
    als_swath = pd.read_parquet(als_outfile)
else:
    als_swath = create_als_swath(extent_gdf, df_ph)
    als_swath = transform_als_swath(als_swath, 
        'EPSG:32614', 
        source_geoid_file=als_geoid_file,
        target_geoid_file=None, 
        input_units='feet', 
        source_datum='nad83'
    )
    als_swath.to_parquet(als_outfile)

In [ ]:
from phoreal.plotting import plot_alongtrack_atl_als3
plot_alongtrack_atl_als3(df_ph, als_swath,'ALS vs ATL08')

### Perform Geolocation Offset

Fine tune the fit between the ICESat-2 and the ALS swath.

In [ ]:
%load_ext autoreload
%autoreload 2

# Call geolocation offset script
from phoreal.geolocation import create_interpolator


df_ph['crosstrack'] = 0


In [ ]:
als_surface_interpolator = create_interpolator(als_swath, grid_resolution=1)

In [ ]:
from phoreal.geolocation import calculate_mae_cost
from scipy.optimize import minimize, brute, fmin


search_grid = (slice(-10, 10, 0.5), slice(-10, 10, 0.5))

result = brute(
    calculate_mae_cost, 
    ranges=search_grid, 
    args=(df_ph.alongtrack[df_ph.atl08_class == 1], 
    df_ph.crosstrack[df_ph.atl08_class == 1], 
    df_ph.h_ph[df_ph.atl08_class == 1], als_surface_interpolator),
    finish=fmin, 
    full_output=True 
)

optimal_shift_at, optimal_shift_xt = result[0]
best_mae = result[1]

print(f"AT shift: {optimal_shift_at:.2f} m")
print(f"XT shift: {optimal_shift_xt:.2f} m")
print(f"Best MAE: {result[1]:.2f} m")

In [ ]:
from phoreal.geolocation import calculate_z_shift


optimal_shift_z = calculate_z_shift(df_ph.alongtrack[df_ph.atl08_class == 1], 
                                    df_ph.crosstrack[df_ph.atl08_class == 1],  
                                    df_ph.h_ph[df_ph.atl08_class == 1], 
                                    optimal_shift_at, 
                                    optimal_shift_xt , 
                                    als_surface_interpolator)



In [ ]:
# apply crosstrack and alongtrack shift to the ALS swath to make the math easier
als_swath['alongtrack'] = als_swath['alongtrack'] - optimal_shift_at
als_swath['crosstrack'] = als_swath['crosstrack'] - optimal_shift_xt




In [ ]:
als_swath = als_swath[als_swath.alongtrack > np.min(df_ph.alongtrack)]
als_swath = als_swath[als_swath.alongtrack < np.max(df_ph.alongtrack)]


In [ ]:
df_ph = df_ph[df_ph.alongtrack > np.min(als_swath.alongtrack)]
df_ph = df_ph[df_ph.alongtrack < np.max(als_swath.alongtrack)]

In [ ]:
import geopandas as gpd

# Define roughly the bounding box for Austin (Min Long, Min Lat, Max Long, Max Lat)
# austin_bbox = (-98.1, 30.1, -97.5, 30.5)

# Load ONLY the buildings within that bounding box from the local file
# Note: Geopandas can read directly from the zipped file!
# austin_buildings = gpd.read_file(file_path, bbox=austin_bbox)

# print(f"Loaded {len(austin_buildings)} buildings in the Austin area.")
austin_building_file = cfg['austin_buildings_file']
#austin_buildings.to_file( austin_building_file,driver='GPKG')

In [ ]:
austin_buildings = gpd.read_file(austin_building_file)
len(austin_buildings)

In [ ]:
%load_ext autoreload
%autoreload 2

from phoreal.als.swath import prepare_icesat2_track
from building_processing import find_intersected_buildings, convert_buildings_to_atxt, remove_complex_intersections

# 1. Prepare data once
gdf_buildings_utm = austin_buildings.to_crs('EPSG:32614')

is2_line_utm, line_x, line_y, line_at_dist = prepare_icesat2_track(
    df_ph, utm_epsg='EPSG:32614', resolution_m=10.0
)

# 2. Pipeline the data through the modular functions
candidates_utm = find_intersected_buildings(
    is2_line_utm, gdf_buildings_utm, buffer_meters=5.0, building_filter_size=300.0
)

# 3. Remove buildings that have complicated building shapes
clean_candidates = remove_complex_intersections(candidates_utm, is2_line_utm)


clean_candidates.to_file(f"austin_bldgs_clean_{atl03_basename}_{gt}.gpkg")


buildings_atxt = convert_buildings_to_atxt(
    clean_candidates, is2_line_utm, line_x, line_y, line_at_dist
)

In [ ]:
data = {
    'name': [f'{atl03_basename}_{gt}'],
    'geometry': [is2_line_utm]
}

# 3. Initialize the GeoDataFrame with a CRS
gdf = gpd.GeoDataFrame(data, crs="EPSG:32614")

# 4. Write to GeoPackage
# Use 'engine="pyogrio"' for high-performance writing
gdf.to_file(f"test_path2.gpkg", layer='layer',driver="GPKG", engine="pyogrio")

In [ ]:
candidates_utm.to_file('candidate_buildings.gpkg', driver='GPKG',index=True)

In [ ]:
%load_ext autoreload
%autoreload 2

from building_processing import filter_grazing_hits

hit_buildings = filter_grazing_hits(buildings_atxt)

In [ ]:
len(hit_buildings)

In [ ]:
# target_building_id = 273324
# target_building_id = 17636
# target_building_id = 159174
# target_building_id = 2479
# target_building_id = 198116
# 12 is good
# 20 exit looks good
target_building_id = hit_buildings.iloc[11].name
target_building = hit_buildings.loc[target_building_id].geometry.buffer(20) 
target_building_atxt = buildings_atxt.loc[target_building_id].geometry.buffer(20) 

In [ ]:
direction = 'exit'

In [ ]:
from building_processing import clip_als_to_buffered_building
target_als = clip_als_to_buffered_building(als_swath, target_building, buffer_m=1.0)
target_als = target_als[np.abs(target_als.crosstrack) < 6]

In [ ]:
cmap = plt.get_cmap('viridis')
# colors = cmap(target_als.h_norm[target_als.classification==6]) 
plt.scatter(target_als.crosstrack, target_als.alongtrack,c='k',label='Non-classified',s=3)
plt.scatter(target_als.crosstrack[target_als.classification==2], target_als.alongtrack[target_als.classification==2],c='orange',s=3,label='Ground')
plt.scatter(target_als.crosstrack[target_als.classification==3], target_als.alongtrack[target_als.classification==3],c='g',s=3,label='Vegetation')
plt.scatter(target_als.crosstrack[target_als.classification==6], target_als.alongtrack[target_als.classification==6],c='cyan',s=3,label='Building')
plt.plot(target_building_atxt.centroid.y, target_building_atxt.centroid.x,'o',label='Centroid')

plt.legend()
plt.title('Nadir view of target building by classification (at/xt)')
plt.xlabel('Crosstrack (m)')
plt.ylabel('Alongtrack (m)')
plt.axis('equal')
plt.show()

In [ ]:
%load_ext autoreload
%autoreload 2
from building_processing import extract_building_edges_2d
edge = extract_building_edges_2d(target_als, target_building_atxt)
edge


In [ ]:
from building_processing import create_line
x_entry,y_entry = create_line(edge['entry']['slope'], edge['entry']['intercept'], x_start=-7, x_stop = 7)
x_exit,y_exit = create_line(edge['exit']['slope'], edge['exit']['intercept'], x_start=-7, x_stop = 7)

In [ ]:
cmap = plt.get_cmap('viridis')
# colors = cmap(target_als.h_norm[target_als.classification==6]) 
plt.scatter(target_als.crosstrack, target_als.alongtrack,c='k',label='Non-classified',s=3)
plt.scatter(target_als.crosstrack[target_als.classification==2], target_als.alongtrack[target_als.classification==2],c='orange',s=3,label='Ground')
plt.scatter(target_als.crosstrack[target_als.classification==3], target_als.alongtrack[target_als.classification==3],c='g',s=3,label='Vegetation')
plt.scatter(target_als.crosstrack[target_als.classification==6], target_als.alongtrack[target_als.classification==6],c='cyan',s=3,label='Building')
plt.plot(x_entry,y_entry,'b--',label = 'Entry')
plt.plot(x_exit,y_exit,'r--',label = 'Exit')
plt.legend()
plt.title('Nadir view of target building by classification (at/xt)')
plt.xlabel('Crosstrack (m)')
plt.ylabel('Alongtrack (m)')
plt.axis('equal')
plt.show()

In [ ]:
plt.scatter(target_als.alongtrack[target_als.classification == 1],target_als.h_norm[target_als.classification == 1],c='gray',s=2,label='Non-classified')
plt.scatter(target_als.alongtrack[target_als.classification == 6],target_als.h_norm[target_als.classification == 6],c='cyan',s=2,label='Building')
plt.scatter(target_als.alongtrack[target_als.classification == 2],target_als.h_norm[target_als.classification == 2],c='orange',s=2,label='Ground')
plt.scatter(target_als.alongtrack[target_als.classification == 3],target_als.h_norm[target_als.classification == 3],c='green',s=2,label='Vegetation')

plt.plot([edge['entry']['intercept'],edge['entry']['intercept']],[-5,20],'b--',label = 'Entry')
plt.plot([edge['exit']['intercept'],edge['exit']['intercept']],[-5,20],'r--',label = 'Exit')
plt.title('Profile view of target building')
plt.xlabel('Alongtrack (m)')
plt.ylabel('Normalized Height (m)')
plt.axis('equal')
plt.show()

In [ ]:
target_ph = df_ph[(edge[direction]['intercept'] + 100) > df_ph.alongtrack] 
target_ph = target_ph[(edge[direction]['intercept'] - 100) < df_ph.alongtrack] 
print(target_ph.columns)
# target_ph = target_ph[target_ph.h_norm < 130]
# target_ph = target_ph[target_ph.h_norm > -10]
len(target_ph)

In [ ]:
plt.scatter(target_als.alongtrack[target_als.classification == 1],target_als.h_norm[target_als.classification == 1],c='gray',s=2,label='Non-classified')
plt.scatter(target_als.alongtrack[target_als.classification == 6],target_als.h_norm[target_als.classification == 6],c='cyan',s=2,label='Building')
plt.scatter(target_als.alongtrack[target_als.classification == 2],target_als.h_norm[target_als.classification == 2],c='orange',s=2,label='Ground')
plt.scatter(target_als.alongtrack[target_als.classification == 3],target_als.h_norm[target_als.classification == 3],c='green',s=2,label='Vegetation')
plt.scatter(target_ph.alongtrack,target_ph.h_norm,c='red',s=2,label='IS2')


plt.plot([edge['entry']['intercept'],edge['entry']['intercept']],[-5,20],'b--',label = 'Entry')
plt.plot([edge['exit']['intercept'],edge['exit']['intercept']],[-5,20],'r--',label = 'Exit')
plt.title('Profile view of target building')
plt.legend()
plt.xlabel('Alongtrack (m)')
plt.ylabel('Normalized Height (m)')
plt.axis('equal')
plt.show()

In [ ]:
plt.scatter(target_als.alongtrack[target_als.classification == 1],target_als.ellip_h[target_als.classification == 1],c='gray',s=2,label='Non-classified')
plt.scatter(target_als.alongtrack[target_als.classification == 6],target_als.ellip_h[target_als.classification == 6],c='cyan',s=2,label='Building')
plt.scatter(target_als.alongtrack[target_als.classification == 2],target_als.ellip_h[target_als.classification == 2],c='orange',s=2,label='Ground')
plt.scatter(target_als.alongtrack[target_als.classification == 3],target_als.ellip_h[target_als.classification == 3],c='green',s=2,label='Vegetation')
plt.scatter(target_ph.alongtrack,target_ph.h_ph,c='red',s=2,label='IS2')

In [ ]:
%load_ext autoreload
%autoreload 2

from building_processing import calculate_orthogonal_distance, classify_photons
from building_processing import classify_photons_als_dynamic
target_ph['crosstrack'] = 0
target_ph = calculate_orthogonal_distance(target_ph, edge[direction], direction, threshold_m=15.0)

edge_als = calculate_orthogonal_distance(target_als, edge[direction], direction, threshold_m=15.0)

# target_ph = classify_photons(target_ph, edge[direction]['roof_median_h'], z_tolerance=1.0)
target_ph = classify_photons_als_dynamic(target_ph, edge_als, z_tolerance=1.0)


In [ ]:
target_ph.attrs

In [ ]:
plt.plot(edge_als.dist_to_wall, 
edge_als.ellip_h,'.',color=[0.6,0.6,0.6])

plt.plot(target_ph.dist_to_wall[target_ph.target_class =='ground'], 
target_ph.h_ph[target_ph.target_class == 'ground'],'.r')
plt.plot(target_ph.dist_to_wall[target_ph.target_class =='roof'], 
target_ph.h_ph[target_ph.target_class == 'roof'],'.b')
# plt.plot(target_ph.dist_to_wall[target_ph.target_class =='noise'], 
# target_ph.h_ph[target_ph.target_class == 'noise'],'.k')


# plt.plot(edge_als.dist_to_wall[edge_als.target_class =='roof'], 
# edge_als.h_norm[edge_als.target_class == 'roof'],'.b')
# plt.plot(edge_als.dist_to_wall[edge_als.target_class =='noise'], 
# edge_als.h_norm[edge_als.target_class == 'noise'],'.k')

In [ ]:
ATL_AGG_CONFIG2 = [
    # Geometry
    {'field': 'x', 'operation': 'median', 'class_field': 'classification', 'class_id': list(range(1,100)), 'outfield': 'x_als'},
    {'field': 'y', 'operation': 'median', 'class_field': 'classification', 'class_id': list(range(1,100)), 'outfield': 'y_als'},

    # Terrain & Bathy
    {'field': 'ortho_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [2], 'outfield': 'als_topo_median'},
    {'field': 'ortho_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [40], 'outfield': 'als_bathy_median'},
    {'field': 'ortho_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [41], 'outfield': 'als_surface_median'},

    # Terrain & Bathy
    {'field': 'ellip_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [2], 'outfield': 'als_topo_ellip_median'},
    {'field': 'ellip_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [40], 'outfield': 'als_bathy_ellip_median'},
    {'field': 'ellip_h', 'operation': 'median', 'class_field': 'classification', 'class_id': [41], 'outfield': 'als_surface_ellip_median'},


    # Canopy
    {'field': 'h_norm', 'operation': 'get_max98', 'class_field': 'classification', 'class_id': [3,4,5], 'outfield': 'als_norm_veg_max98'},
]

In [ ]:
df_seg = analysis.aggregate_by_segment(df_ph, ATL08_AGG_CONFIG, res=30)

In [ ]:
df_seg.columns

In [ ]:
als_seg = analysis.aggregate_by_segment(als_swath, ALS_AGG_CONFIG, res=30)

In [ ]:
als_seg.columns

In [ ]:
als_fwhm = als_swath[np.abs(als_swath['crosstrack']) < (7.55/2)]

In [ ]:
als_fwhm.head()

In [ ]:
als_seg2 = analysis.aggregate_by_segment(als_fwhm, ALS_AGG_CONFIG, res=30)

In [ ]:
als_seg.columns

In [ ]:
err1_veg = als_seg.als_norm_veg_max98 - df_seg.h_canopy
print(np.abs(np.mean(err1_veg)))
print(np.sqrt(np.mean((err1_veg**2))))
# err1_terrain = als_seg.als_surface_ellip_median - df.h

In [ ]:
err2_veg = als_seg2.als_norm_veg_max98 - df_seg.h_canopy
print(np.abs(np.mean(err2_veg)))
print(np.sqrt(np.mean((err2_veg**2))))

In [ ]:
err1_ter = als_seg.als_topo_ellip_median - df_seg.h_te_median
print(np.abs(np.mean(err1_ter)))
print(np.sqrt(np.mean((err1_ter**2))))


In [ ]:
err2_ter = als_seg2.als_topo_ellip_median - df_seg.h_te_median
print(np.abs(np.mean(err2_ter)))
print(np.sqrt(np.mean((err2_ter**2))))

In [ ]:
# from building_processing import clip_als_to_buffered_building
# from building_processing import extract_building_edges_2d


# # target_building = hit_buildings.loc[target_building_id].geometry.buffer(20) 
# for target_building_id in hit_buildings.index:
#     print(target_building_id)
#     target_building = hit_buildings.loc[target_building_id].geometry.buffer(20) 
#     target_atxt = buildings_atxt.loc[target_building_id].geometry.buffer(20) 
#     target_als = clip_als_to_buffered_building(als_swath, target_building, buffer_m=1.0)
#     target_als = target_als[np.abs(target_als.crosstrack) < 7] 

#     edge = extract_building_edges_2d(target_als, target_atxt)
#     for direction in ['entry','exit']:
#         if edge[direction]['valid']:
            

#             out_name = f'{atl03_basename}_{gt}_{target_building_id}_{direction}'
#             print(out_name)

#             intercept = edge[direction]['intercept']
#             target_ph = df_ph[df_ph.alongtrack.between(intercept - 50, intercept + 50)]

#             from building_processing import calculate_orthogonal_distance, classify_photons
#             target_ph = calculate_orthogonal_distance(target_ph, edge[direction], direction, threshold_m=10.0)

#             target_ph = classify_photons(target_ph, edge[direction]['roof_median_h'], z_tolerance=1.0)

#             target_ph.to_parquet(f'outputs/{out_name}_out.pqt')
